# Training, compiling, kernels, dynamics and postselection in qg terms (§159–§162, §164)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viny2030/qang/blob/main/notebooks/qang_methods.ipynb)

Five short studies, each with its published, pre-registered result (RESEARCH_NOTES):

| § | question | answer |
|---|---|---|
| 159 | does training in probability (qg) coordinates help? | no: the angle is the Fisher-flat coordinate; qg is the read-out; the natural-gradient metric is qg_YY correlators |
| 160 | how should fractional gates be compiled? | by minimal pulse area: the error follows the area |
| 161 | how to repair a noisy quantum-kernel Gram matrix? | the qg filter beats the diagonal normalization |
| 162 | variational time evolution against Trotter? | fixed depth wins at late times; every channel is an affine map of qg values |
| 164 | time loops and black holes? | teleportation with postselection: P-CTC success cos²(φ/2), a working black-hole decoder |

Local simulators of IBM devices; about 5 minutes.

## 1. Install

In [1]:
import importlib, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "qang", "qiskit-aer", "qiskit-ibm-runtime", "scikit-learn"], check=False)
importlib.invalidate_caches()
try:
    import qang.postselection
    ok = hasattr(qang.postselection, "yk_decoder_circuit")
except ImportError:
    ok = False
if not ok:  # the published qang predates these sections: install from GitHub
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                    "git+https://github.com/Viny2030/qang.git"], check=True)
    print("installed qang from GitHub; if an import fails below, restart the runtime once")
import qang
print("qang", qang.__version__)

qang 0.11.0


## 2. Set-up

In [2]:
import json, math, os, time, urllib.request
import numpy as np
import qiskit_ibm_runtime.fake_provider as fp
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

T0 = time.time()


def run(circuits, backend_name="FakeFez", shots=4000, seed=1):
    """Transpile to a fake IBM device and sample with its noise model; returns the list of counts."""
    b = getattr(fp, backend_name)()
    ts = transpile(circuits, b, optimization_level=1, seed_transpiler=137)
    res = AerSimulator.from_backend(b).run(ts, shots=shots, seed_simulator=seed).result()
    return [res.get_counts(i) for i in range(len(circuits))]


def repo_json(name):
    local = os.path.join("..", "examples", "data", name)
    if os.path.exists(local):
        return json.load(open(local, encoding="utf-8"))
    url = f"https://raw.githubusercontent.com/Viny2030/qang/main/examples/data/{name}"
    return json.loads(urllib.request.urlopen(url).read().decode("utf-8"))

from qang import fisher as F, gradients as GR, pulse_area as PA, kernels as KR, channels as CH, varqte as VQ
from qang import postselection as PS, ibm

## 3. The angle is the Fisher-flat coordinate (§159)

Per shot the Fisher information is 1 in the angle and 1/(1 − qg²) in qg_Z. One evaluation cannot give a derivative (two functions a + b cos θ + c sin θ agree at θ + s and differ in E′(θ)), so 2 per parameter is the minimum in any coordinate. In a real RY + CZ ansatz every metric diagonal is exactly 1/4 and the off-diagonal terms are qg_Y correlators. Published (TFIM n = 4, 20 seeds): descent in qg coordinates never converges; the qg_YY block metric is 1.3× faster than descent in θ; the full metric 4.6×. Below, 3 seeds and the published learning rates.

In [3]:
f1, f2, d1, d2 = F.indistinguishable_pair(0.7, math.pi / 2)
print(f"one evaluation: f1 = {f1(0.7 + math.pi / 2):+.4f}, f2 = {f2(0.7 + math.pi / 2):+.4f}; derivatives {d1:+.3f} vs {d2:+.3f}")
n, L = 4, 3
H = F.tfim_hamiltonian(n)
E0 = float(np.linalg.eigvalsh(H)[0])
th = np.random.default_rng(1).uniform(0, 2 * math.pi, n * L)
print("metric diagonal:", np.round(np.diag(F.full_metric(th, n, L)), 6)[:4], "...")


def steps_to_tol(method, lr, seed, tol=0.03, max_steps=150):
    th = np.random.default_rng(seed).uniform(0, 2 * math.pi, n * L)
    q, s = np.cos(th), np.where(np.sin(th) >= 0, 1, -1)
    for k in range(max_steps + 1):
        if F.energy(th, H, n, L) - E0 <= tol:
            return k
        g = F.gradient(th, H, n, L)
        if method == "GD-theta":
            th = th - lr * g
        elif method == "GD-qg":
            for i in range(th.size):
                q[i], s[i] = GR.signed_qg_step(q[i], int(s[i]), g[i], lr)
            th = np.array([GR.signed_theta(q[i], int(s[i])) for i in range(th.size)])
        else:
            G = F.block_metric(th, n, L) if method == "QNG-block" else F.full_metric(th, n, L)
            th = th - lr * np.linalg.solve(G + 1e-3 * np.eye(th.size), g)
    return f">{max_steps}"


for m, lr in (("GD-theta", 0.2), ("GD-qg", 0.005), ("QNG-block", 0.05), ("QNG-full", 0.1)):
    print(f"{m:9s} lr {lr:5}: steps to E - E0 <= 0.03 for seeds 6, 7, 8: {[steps_to_tol(m, lr, s) for s in (6, 7, 8)]}")

one evaluation: f1 = -0.6442, f2 = -0.6442; derivatives -0.644 vs +0.765
metric diagonal: [0.25 0.25 0.25 0.25] ...


GD-theta  lr   0.2: steps to E - E0 <= 0.03 for seeds 6, 7, 8: ['>150', 72, 105]


GD-qg     lr 0.005: steps to E - E0 <= 0.03 for seeds 6, 7, 8: ['>150', '>150', '>150']


QNG-block lr  0.05: steps to E - E0 <= 0.03 for seeds 6, 7, 8: ['>150', 63, 48]


QNG-full  lr   0.1: steps to E - E0 <= 0.03 for seeds 6, 7, 8: [22, 19, 23]


## 4. Minimal pulse area (§160)

A random amplitude error damps qg_Z as cos θ · e^(−θ²σ²/2); the shortest representative wins. RX has period 2π (global phase), RZZ period π (the difference Z⊗Z is two virtual RZ). `minimal_area_circuit` preserves the unitary exactly. Published (QAOA model, 150 instances): the 2π wrap cuts the error 4.1×, the period-π RZZ wrap another 1.14×.

In [4]:
from qiskit.quantum_info import Operator
for th in (1.0, 3.0, 5.0):
    print(f"theta {th}: qg_Z damped {PA.rx_qg_z_random_amplitude(th, 0.05):+.4f}, shortest representative "
          f"{PA.minimal_area_angle(th)[0]:+.3f} -> {PA.rx_qg_z_random_amplitude(PA.minimal_area_angle(th)[0], 0.05):+.4f}")
qc = QuantumCircuit(4)
rng = np.random.default_rng(160)
for g, b in zip(rng.uniform(0, 2 * math.pi, 2), rng.uniform(0, 2 * math.pi, 2)):
    for a in range(4):
        qc.rzz(2 * g, a, (a + 1) % 4)
    for a in range(4):
        qc.rx(2 * b, a)
m = PA.minimal_area_circuit(qc)
print(f"QAOA: summed area {PA.area(qc):.2f} -> {PA.area(m):.2f} rad; same unitary: "
      f"{np.allclose(Operator(m).data, Operator(qc).data)}")

theta 1.0: qg_Z damped +0.5396, shortest representative +1.000 -> +0.5396
theta 3.0: qg_Z damped -0.9789, shortest representative +3.000 -> -0.9789
theta 5.0: qg_Z damped +0.2749, shortest representative -1.283 -> +0.2831
QAOA: summed area 107.75 -> 21.67 rad; same unitary: True


## 5. Repairing a quantum-kernel Gram matrix (§161)

The weight-1 fidelity kernel K = (x·y)² from the inversion test conserves the Hamming weight, so the qg filter applies; the usual repair divides by the diagonal, which costs extra circuits. Published (40 + 40 points, 3 devices): the filter repairs the Gram matrix 2.4–6× and beats the normalization; accuracy barely moves. Here 20 + 20 points on FakeFez.

In [5]:
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.svm import SVC
X, y = load_breast_cancer(return_X_y=True)
rng = np.random.default_rng(161)
tr = np.concatenate([rng.choice(np.where(y == c)[0], 10, replace=False) for c in (0, 1)])
rest = np.setdiff1d(np.arange(len(y)), tr)
te = np.concatenate([rng.choice(rest[y[rest] == c], 10, replace=False) for c in (0, 1)])
mu, sd = X[tr].mean(0), X[tr].std(0)
p = PCA(4).fit((X[tr] - mu) / sd)
A, Bt = (KR.unary_embedding(p.transform((X[i] - mu) / sd)) for i in (tr, te))
ntr, nte = len(tr), len(te)
pairs = [(A[i], A[j]) for i in range(ntr) for j in range(ntr)] + [(Bt[i], A[j]) for i in range(nte) for j in range(ntr)]
pairs += [(Bt[i], Bt[i]) for i in range(nte)]
est = [KR.kernel_from_counts(c) for c in run([ibm.unary_overlap_circuit(a, b) for a, b in pairs], shots=2000)]
K0tr, K0te = KR.exact_kernel(A, A), KR.exact_kernel(Bt, A)
acc = lambda Ktr, Kte: int((SVC(kernel="precomputed").fit(Ktr, y[tr]).predict(Kte) == y[te]).sum())
iu = np.triu_indices(ntr, 1)
print(f"exact kernel: {acc(K0tr, K0te)}/{nte}")
for kind in ("raw", "filtered"):
    v = np.array([e[kind] for e in est])
    Ktr, Kte, dte = v[:ntr * ntr].reshape(ntr, ntr), v[ntr * ntr:ntr * ntr + nte * ntr].reshape(nte, ntr), v[-nte:]
    dtr = np.diag(Ktr)
    for label, (Kt, Ke) in ((kind, (Ktr, Kte)), (kind + " normalized", (KR.normalize_gram(Ktr, dtr, dtr), KR.normalize_gram(Kte, dte, dtr)))):
        print(f"{label:20s} Gram error {np.abs(Kt[iu] - K0tr[iu]).mean():.4f}   accuracy {acc(Kt, Ke)}/{nte}   mean diagonal {dtr.mean():.3f}")

exact kernel: 17/20
raw                  Gram error 0.0328   accuracy 17/20   mean diagonal 0.955
raw normalized       Gram error 0.0227   accuracy 17/20   mean diagonal 0.955
filtered             Gram error 0.0103   accuracy 17/20   mean diagonal 1.000
filtered normalized  Gram error 0.0103   accuracy 17/20   mean diagonal 1.000


## 6. Channels as qg maps, and fixed-depth time evolution (§162)

Every channel acts on (qg_X, qg_Y, qg_Z) as r′ = M r + c. McLachlan variational evolution with a weight-preserving 6-layer ansatz keeps the circuit depth fixed (48 two-qubit gates) while Trotter grows. Published (XXZ, n = 5, two excitations): at t = 2 the variational circuit beats the best Trotter circuit on three devices. The angles θ(t) are read from the repository (integrating them takes minutes).

In [6]:
r = np.array([0.3, -0.2, 0.5])
for name, k in (("amplitude damping 0.3", CH.amplitude_damping(0.3)), ("phase damping 0.4", CH.phase_damping(0.4)),
                ("depolarizing 0.2", CH.depolarizing(0.2)), ("T1/T2 t=1, 3, 2.5", CH.thermal_relaxation(1, 3, 2.5))):
    M, c = CH.bloch_affine(k)
    print(f"{name:22s} qg vector {r} -> {np.round(M @ r + c, 4)}")
d = repo_json("varqte_kraus_qg.json")
h = np.random.default_rng(162).uniform(-0.5, 0.5, 5)
ex = ibm.xxz_exact_occupations(5, 1.0, 2.0, h, 2.0, (0, 1))
var = VQ.ansatz(np.array(d["thetas"]["2.0"]), 6, measure=True)
trot = ibm.xxz_trotter(5, 1.0, 2.0, h, 2.0, 4, (0, 1), order=2)
for name, c in zip(("variational", "Trotter r = 4"), run([var, trot])):
    print(f"t = 2, {name:13s}: error raw {np.abs(VQ.occupations_from_counts(c, 5) - ex).mean():.4f}, "
          f"filtered {np.abs(VQ.occupations_from_counts(c, 5, 2) - ex).mean():.4f}")

amplitude damping 0.3  qg vector [ 0.3 -0.2  0.5] -> [ 0.251  -0.1673  0.65  ]
phase damping 0.4      qg vector [ 0.3 -0.2  0.5] -> [ 0.2324 -0.1549  0.5   ]
depolarizing 0.2       qg vector [ 0.3 -0.2  0.5] -> [ 0.24 -0.16  0.4 ]
T1/T2 t=1, 3, 2.5      qg vector [ 0.3 -0.2  0.5] -> [ 0.2011 -0.1341  0.6417]


t = 2, variational  : error raw 0.0332, filtered 0.0168
t = 2, Trotter r = 4: error raw 0.0606, filtered 0.0218


## 7. Time loops and black holes as postselected teleportation (§164)

**P-CTC:** a qubit sent to its own past is half of a Bell pair projected back on it. Rotated by RX(φ), the postselection succeeds with probability cos²(φ/2), whose qang angle is φ; the grandfather paradox (φ = π) has probability 0. **Yoshida–Kitaev:** a message falls into a 3-qubit black hole; with a scrambling Clifford, postselecting a radiated qubit and its partner on a Bell state recovers the message (success 1/4, fidelity 1 noiseless). Published (3 devices): the decoder gives fidelity 0.55–0.74 > 1/2. These are information models, not statements about spacetime.

In [7]:
phis = [0, math.pi / 3, math.pi / 2, 2 * math.pi / 3, math.pi]
cs = run([PS.pctc_circuit(p, False) for p in phis] + [PS.pctc_circuit(p, True) for p in phis], shots=8000)
print("phi     success (exact)    qang angle of success   controlled qg_Z(S) (exact)")
for i, p in enumerate(phis):
    s = PS.pctc_from_counts(cs[i])["success"]
    z = PS.pctc_from_counts(cs[5 + i])["qg_z_S"]
    print(f"{p:.3f}   {s:.4f} ({PS.pctc_success(p):.4f})    {math.acos(min(1, max(-1, 2 * s - 1))):.3f}                   "
          f"{z:+.4f} ({PS.pctc_controlled_qg_z(p):+.4f})")
scr = PS.find_scrambler()
for name, c in zip(("scrambler", "identity"), run([PS.yk_decoder_circuit(scr), PS.yk_decoder_circuit(QuantumCircuit(3))], shots=8000)):
    r = PS.yk_from_counts(c)
    print(f"decoder with {name:9s}: success {r['success']:.3f}, fidelity of the recovered pair {r['fidelity']:.3f}")
print(f"total time {time.time() - T0:.0f} s")

phi     success (exact)    qang angle of success   controlled qg_Z(S) (exact)
0.000   0.9826 (1.0000)    0.264                   -0.0036 (+0.0000)
1.047   0.7374 (0.7500)    1.076                   +0.1455 (+0.1429)
1.571   0.4943 (0.5000)    1.582                   +0.3129 (+0.3333)
2.094   0.2485 (0.2500)    2.098                   +0.5809 (+0.6000)
3.142   0.0090 (0.0000)    2.952                   +0.9605 (+1.0000)


decoder with scrambler: success 0.245, fidelity of the recovered pair 0.738
decoder with identity : success 0.994, fidelity of the recovered pair 0.252
total time 113 s


## Summary

* Optimize in the angle, read out in qg; the natural-gradient metric is a set of qg_YY correlators (§159).
* Compile fractional gates by minimal area; the qg damping cos θ e^(−θ²σ²/2) is the error budget (§160).
* For a weight-preserving kernel the qg filter is the better Gram-matrix repair (§161).
* Every channel is an affine map of qg values; fixed-depth variational evolution beats Trotter at late times (§162).
* Postselection is a sector filter: P-CTCs and black-hole decoding are teleportation with it (§164).